In [1]:
import lodat as lo

In [3]:
import pandas as pd
path_to_big_data = r"C:\LODAT\test_assets\large_data.csv"
df = pd.read_csv(path_to_big_data, index_col=0)

In [21]:
print(df.shape)

(4608000, 6)


In [22]:
df.head()

,Look,Depression,Twist,Frequency,Polarization,RCS
0,0.092692,2.055946,7.821476,2000,HH,-4.088469
1,-0.049396,2.001348,-3.037385,2000,HH,-5.566499
2,-0.083841,3.754457,6.129944,2000,HH,-8.347156
3,-0.070949,3.089924,6.610768,2000,HH,-6.338199
4,0.130640,3.371985,-8.729374,2000,HH,3.344893


Single slice

In [7]:
# Input parameters
import random
f = random.choice(df.Frequency.unique())
p = random.choice(df.Polarization.unique())
d0, d1 = (0, 5)
l0, l1 = (0.5, 1.5)

In [24]:
# Normal slicing
vector_mask = (df.Frequency == float(f)) & (df.Polarization == p)
depr_mask = (df.Depression >= d0) & (df.Depression < d1)
look_mask = (df.Look >= l0) & (df.Look < l1)
mask = vector_mask & depr_mask & look_mask
result1 = df[mask]
result1.head()

,Look,Depression,Twist,Frequency,Polarization,RCS
84,0.511329,1.638793,-3.012399,9000,HH,-3.538765
86,0.512114,2.976423,2.611331,9000,HH,-7.592221
90,0.598733,2.139616,-3.926326,9000,HH,1.914282
98,0.509012,2.281016,0.188420,9000,HH,0.353330
100,0.654237,2.531809,-6.035132,9000,HH,-5.260621


In [25]:
# Slicing as numpy array
vector_mask = (df.Frequency.values == float(f)) & (df.Polarization.values == p)
depr_mask = (df.Depression.values >= d0) & (df.Depression.values < d1)
look_mask = (df.Look.values >= l0) & (df.Look.values < l1)
mask = vector_mask & depr_mask & look_mask
result2 = df[mask]
result2.head()

,Look,Depression,Twist,Frequency,Polarization,RCS
84,0.511329,1.638793,-3.012399,9000,HH,-3.538765
86,0.512114,2.976423,2.611331,9000,HH,-7.592221
90,0.598733,2.139616,-3.926326,9000,HH,1.914282
98,0.509012,2.281016,0.188420,9000,HH,0.353330
100,0.654237,2.531809,-6.035132,9000,HH,-5.260621


NOTE: using .values and slicing as a numpy array cuts the time in half

In [1]:
# Using dask
import dask.dataframe as dd

In [4]:
ddf = dd.read_csv(path_to_big_data)
ddf = ddf.set_index('Unnamed: 0')

In [5]:
ddf.head()

,Look,Depression,Twist,Frequency,Polarization,RCS
Unnamed: 0,,,,,,
0,0.083859,2.076525,1.816321,8000,VV,-5.315920
0,-0.014236,2.884426,-4.245023,2000,VV,1.653626
0,-0.048339,0.982048,-2.160909,14000,VV,3.396975
0,0.047258,-7.724052,0.518514,3000,HH,0.402661
0,-0.076953,1.759516,-4.111208,10000,HH,-8.198989


In [65]:
ddf.map_partitions(len).compute()

0    748224
1    628864
2    978112
3    786624
4    714176
5    752000
dtype: int64

In [8]:
# Slicing using Dask
vector_mask = (ddf.Frequency == float(f)) & (ddf.Polarization == p)
depr_mask = (ddf.Depression >= d0) & (ddf.Depression < d1)
look_mask = (ddf.Look >= l0) & (ddf.Look < l1)
mask = vector_mask & depr_mask & look_mask
tmp = ddf.mask(mask)
# result3 = result3.compute()
# result3.head()

In [10]:
tmp.RCS.mean().compute()

1.4716103423522529

Performing many slices

In [50]:
pd.set_option('display.float_format', lambda x: '%.3f' % x)
df.describe()

,Look,Depression,Twist,Frequency,RCS
count,4608000.000,4608000.000,4608000.000,4608000.000,4608000.000
mean,179.997,-2.500,0.005,9500.000,1.472
std,103.923,5.025,5.001,4609.773,8.376
min,-0.294,-10.094,-25.631,2000.000,-25.013
25%,89.997,-7.500,-3.365,5750.000,-2.533
50%,179.999,-2.483,0.006,9500.000,0.868
75%,269.999,2.500,3.379,13250.000,4.314
max,360.371,5.120,27.135,17000.000,117.765


In [11]:
freqs = df.Frequency.unique()
pols = df.Polarization.unique()
looks = [(lk-0.5, lk+0.5) for lk in range(30, 180, 1)]
deprs = [(-10, -5), (0, 5)]

In [12]:
from itertools import product
params = list(product(freqs, pols, looks, deprs))
print('Total number of iterations', len(params))

Total number of iterations 9600


In [15]:
# Standard for loop
rcs = []
for param in params:
    f, p, (l0, l1), (d0, d1) = param

    vector_mask = (df.Frequency.values == float(f)) & (df.Polarization.values == p)
    depr_mask = (df.Depression.values >= d0) & (df.Depression.values < d1)
    look_mask = (df.Look.values >= l0) & (df.Look.values < l1)
    mask = vector_mask & depr_mask & look_mask

    tmp = df[mask]
    rcs.append(tmp.RCS.mean())

In [13]:
# Dask loop
rcs2 = []
for param in params:
    f, p, (l0, l1), (d0, d1) = param
    vector_mask = (ddf.Frequency == float(f)) & (ddf.Polarization == p)
    depr_mask = (ddf.Depression >= d0) & (ddf.Depression < d1)
    look_mask = (ddf.Look >= l0) & (ddf.Look < l1)
    mask = vector_mask & depr_mask & look_mask
    tmp = ddf.mask(mask)
    rcs2.append(tmp.RCS.mean())

In [14]:
import dask
my_results = dask.compute(rcs2)

In [16]:
df.head()

,Look,Depression,Twist,Frequency,Polarization,RCS
0,0.092692,2.055946,7.821476,2000,HH,-4.088469
1,-0.049396,2.001348,-3.037385,2000,HH,-5.566499
2,-0.083841,3.754457,6.129944,2000,HH,-8.347156
3,-0.070949,3.089924,6.610768,2000,HH,-6.338199
4,0.130640,3.371985,-8.729374,2000,HH,3.344893


In [17]:
df.groupby('Frequency')

In [20]:
for freq, fdf in df.groupby(['Frequency', 'Polarization']):
    print(freq, fdf.shape)

(2000, 'HH') (144000, 6)
(2000, 'VV') (144000, 6)
(3000, 'HH') (144000, 6)
(3000, 'VV') (144000, 6)
(4000, 'HH') (144000, 6)
(4000, 'VV') (144000, 6)
(5000, 'HH') (144000, 6)
(5000, 'VV') (144000, 6)
(6000, 'HH') (144000, 6)
(6000, 'VV') (144000, 6)
(7000, 'HH') (144000, 6)
(7000, 'VV') (144000, 6)
(8000, 'HH') (144000, 6)
(8000, 'VV') (144000, 6)
(9000, 'HH') (144000, 6)
(9000, 'VV') (144000, 6)
(10000, 'HH') (144000, 6)
(10000, 'VV') (144000, 6)
(11000, 'HH') (144000, 6)
(11000, 'VV') (144000, 6)
(12000, 'HH') (144000, 6)
(12000, 'VV') (144000, 6)
(13000, 'HH') (144000, 6)
(13000, 'VV') (144000, 6)
(14000, 'HH') (144000, 6)
(14000, 'VV') (144000, 6)
(15000, 'HH') (144000, 6)
(15000, 'VV') (144000, 6)
(16000, 'HH') (144000, 6)
(16000, 'VV') (144000, 6)
(17000, 'HH') (144000, 6)
(17000, 'VV') (144000, 6)
